# Agents Functionality Test

Smoke-test every implemented specialist agent in mock mode:

- `InformationAgent`: metrics and anomaly detection
- `KnowledgeAgent`: business definitions and context
- `MetadataAgent`: ownership, DQ scores, lineage
- `CapacityAgent`: Jira issue read and ticket creation
- `RuleAgent`: list, create, and evaluate rules

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_root = next((p for p in [cwd, *cwd.parents] if (p / 'src').exists()), cwd)
sys.path.insert(0, str(project_root / 'src'))
print('project_root:', project_root)

In [ ]:
from config.settings import AppConfig
from core.base_agent import AgentRequest
from agents.information_agent import InformationAgent
from agents.knowledge_agent import KnowledgeAgent
from agents.metadata_agent import MetadataAgent
from agents.capacity_agent import CapacityAgent
from agents.rule_agent import RuleAgent, RULE_REGISTRY

config = AppConfig()
agents = {
    'information': InformationAgent(config=config, enable_mock=True),
    'knowledge': KnowledgeAgent(config=config, enable_mock=True),
    'metadata': MetadataAgent(config=config, enable_mock=True),
    'capacity': CapacityAgent(config=config, enable_mock=True),
    'rule': RuleAgent(config=config, enable_mock=True),
}

for name, agent in agents.items():
    print(name, agent.health_check())

In [ ]:
def run_agent(name, query, intent='', products=None, context=None):
    request = AgentRequest(
        query=query,
        intent=intent,
        data_products=products or [],
        context=context or {},
        time_range='last_month',
    )
    result = agents[name].execute(request)
    print('\n' + '=' * 80)
    print(name.upper(), 'success=', result.success, 'confidence=', result.confidence)
    print('sources:', result.sources)
    print('metadata:', result.metadata)
    print(result.summary[:1200])
    assert result.success
    return result

info = run_agent('information', 'Why did retention drop last month?', products=['retention'])
knowledge = run_agent('knowledge', 'What is GRR and how is it calculated?')
metadata = run_agent('metadata', 'Who owns the retention dataset?', products=['retention'])
capacity_read = run_agent('capacity', 'Show open Jira bugs for retention', products=['retention'])
rule_list = run_agent('rule', 'List all data quality rules')

In [ ]:
# Capacity write path: mock Jira ticket creation.
ticket = run_agent(
    'capacity',
    'Create ticket for retention completeness issue',
    products=['retention'],
    context={
        'ticket_summary': 'Notebook test ticket',
        'ticket_description': 'Created by agents functionality notebook.',
        'priority': 'High',
    },
)
print('ticket_id:', ticket.data.get('ticket_id'))
assert ticket.data.get('ticket_id')

In [ ]:
# Rule create and evaluate paths.
before = len(RULE_REGISTRY)
created = run_agent(
    'rule',
    'Create rule for retention freshness threshold',
    products=['retention'],
    context={
        'rule_name': 'Notebook Retention Freshness Check',
        'asset': 'analytics.retention_metrics',
        'expression': 'last_refresh_hours < 24',
        'threshold': 24,
        'severity': 'Medium',
    },
)
after = len(RULE_REGISTRY)
print('rules before:', before, 'after:', after)
assert after == before + 1

evaluation = run_agent('rule', 'Evaluate retention rules', products=['retention'])
assert isinstance(evaluation.data, list)